# Object Counting in Video

**YOLO Multi-Task Computer Vision Assignment — Task 2**

## Objective

Detect, track and count people crossing a defined line using
pretrained YOLO26n, ByteTrack, OpenCV and Ultralytics ObjectCounter.

Two video experiments were executed and saved.
No custom model training was required.


## Execution and reproducibility

Use Google Colab with Google Drive mounted for the saved inputs and outputs. Install dependencies using the setup cells. The notebook contains experimental steps that may download media, run inference or train a model; rerun only the stages you intend and ensure required input paths exist. Results were produced in the documented Colab environment; a clean end-to-end rerun on a fresh runtime has not been certified.

Notebook outputs are cleared for safe version control. Results and limitations are documented below and in `README.md`.


In [ ]:

from pathlib import Path
import subprocess
import sys
import torch
from google.colab import drive

# 1. Mount Google Drive
drive.mount("/content/drive")

# 2. Restore project from GitHub
ROOT = Path("/content/yolo-multi-task-computer-vision")

if not (ROOT / ".git").is_dir():
    assert not ROOT.exists(), "Inspect existing project folder"
    subprocess.run([
        "git", "clone",
        "https://github.com/Ibraheim11/yolo-multi-task-computer-vision.git",
        str(ROOT)
    ], check=True)
else:
    subprocess.run([
        "git", "-C", str(ROOT),
        "pull", "--ff-only", "origin", "main"
    ], check=True)

sys.path.insert(0, str(ROOT))

# 3. Prepare persistent output folder
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/object_counting"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 4. Check Ultralytics
try:
    import ultralytics
    from ultralytics import YOLO, solutions
    print("Ultralytics:", ultralytics.__version__)
except ImportError:
    print("Ultralytics missing; installation needed")

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Repository:", ROOT)
print("Outputs:", OUTPUT_DIR)
print("TASK 2 SETUP CHECK COMPLETE")


In [ ]:
%pip install "ultralytics==8.4.208" "opencv-python-headless" "lap"

In [ ]:
import torch
import cv2
import ultralytics
from ultralytics import YOLO, solutions

print("========== TASK 2 ENVIRONMENT ==========")
print("Ultralytics:", ultralytics.__version__)
print("OpenCV:", cv2.__version__)
print("GPU:", torch.cuda.get_device_name(0))

assert ultralytics.__version__ == "8.4.208"
assert torch.cuda.is_available()
assert hasattr(solutions, "ObjectCounter")

print("\nOBJECT COUNTING ENVIRONMENT: PASS")

In [ ]:
from pathlib import Path
import requests
import cv2
from IPython.display import display, Image

VIDEO_DIR = OUTPUT_DIR / "videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_PATH = VIDEO_DIR / "solutions_ci_demo.mp4"

VIDEO_URL = (
    "https://github.com/ultralytics/assets/releases/"
    "download/v0.0.0/solutions_ci_demo.mp4"
)

# Download only if missing
if not VIDEO_PATH.is_file():
    response = requests.get(VIDEO_URL, timeout=120)
    response.raise_for_status()
    VIDEO_PATH.write_bytes(response.content)

# Check video
cap = cv2.VideoCapture(str(VIDEO_PATH))

assert cap.isOpened(), "Video could not be opened"

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

success, frame = cap.read()
cap.release()

assert success, "Cannot read first frame"
assert width > 0 and height > 0 and fps > 0

# Draw a proposed horizontal counting line
line_y = height // 2
REGION = [(0, line_y), (width - 1, line_y)]

cv2.line(
    frame,
    REGION[0],
    REGION[1],
    (0, 0, 255),
    3
)

cv2.putText(
    frame,
    "COUNTING LINE",
    (15, max(25, line_y - 15)),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.7,
    (0, 0, 255),
    2
)

print("===== VIDEO INFORMATION =====")
print("Video:", VIDEO_PATH)
print("Resolution:", width, "x", height)
print("FPS:", fps)
print("Total frames:", frames)
print("Counting region:", REGION)

display(Image(data=cv2.imencode(".jpg", frame)[1].tobytes()))

print("\nVIDEO PREPARATION: PASS")

In [ ]:

from ultralytics import solutions
from pathlib import Path
import cv2
import json
import torch

# Input video and output files
assert VIDEO_PATH.is_file()

RUN_DIR = OUTPUT_DIR / "person_counting_demo"
RUN_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = RUN_DIR / "counted_people.mp4"
REPORT = RUN_DIR / "counting_results.json"

assert not OUTPUT_VIDEO.exists(), "Output already exists; inspect it first"
assert not REPORT.exists(), "Report already exists; inspect it first"

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "Cannot open video"

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

assert width > 0 and height > 0 and fps > 0

REGION = [(0, height // 2), (width - 1, height // 2)]

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

assert writer.isOpened(), "Cannot create output video"

counter = solutions.ObjectCounter(
    model="yolo26n.pt",
    region=REGION,
    tracker="bytetrack.yaml",
    classes=[0],  # Person only
    conf=0.25,
    device=0 if torch.cuda.is_available() else "cpu",
    show=False,
    verbose=False
)

frames = 0
last_result = None

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        last_result = counter(frame)
        writer.write(last_result.plot_im)
        frames += 1

finally:
    cap.release()
    writer.release()

assert frames > 0
assert OUTPUT_VIDEO.is_file()
assert OUTPUT_VIDEO.stat().st_size > 0
assert last_result is not None

summary = {
    "model": "yolo26n.pt",
    "tracker": "bytetrack.yaml",
    "target_class": "person",
    "video": str(VIDEO_PATH),
    "processed_frames": frames,
    "line": REGION,
    "in_count": int(last_result.in_count),
    "out_count": int(last_result.out_count),
    "classwise_count": dict(last_result.classwise_count),
    "output_video": str(OUTPUT_VIDEO)
}

REPORT.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("\n========== OBJECT COUNTING RESULTS ==========")
print("Processed frames:", frames)
print("IN:", summary["in_count"])
print("OUT:", summary["out_count"])
print("TOTAL crossings:", summary["in_count"] + summary["out_count"])
print("Output video:", OUTPUT_VIDEO)
print("Report:", REPORT)
print("\nOBJECT COUNTING: PASS")


In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Video, display

SOURCE = OUTPUT_DIR / "person_counting_demo/counted_people.mp4"
PREVIEW = OUTPUT_DIR / "person_counting_demo/counted_people_h264.mp4"

assert SOURCE.is_file(), "Counting video not found"

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(SOURCE),
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "23",
        "-pix_fmt", "yuv420p",
        "-an",
        str(PREVIEW)
    ],
    check=True,
    capture_output=True,
    text=True
)

assert PREVIEW.is_file() and PREVIEW.stat().st_size > 0

print("Video saved:", PREVIEW)
display(Video(str(PREVIEW), embed=True))

In [ ]:
from pathlib import Path
import json

RUN_DIR = OUTPUT_DIR / "person_counting_demo"

required = {
    "original_video": OUTPUT_DIR / "videos/solutions_ci_demo.mp4",
    "counted_video": RUN_DIR / "counted_people.mp4",
    "preview_video": RUN_DIR / "counted_people_h264.mp4",
    "results": RUN_DIR / "counting_results.json",
}

for name, path in required.items():
    assert path.is_file(), f"Missing: {path}"
    print(f"✅ {name}: {path.stat().st_size / 1024**2:.2f} MB")

data = json.loads(required["results"].read_text())

assert data["processed_frames"] == 62
print("\nSaved counts:", data["in_count"], "IN /", data["out_count"], "OUT")
print("OBJECT COUNTING DEMO SAVED: PASS")

In [ ]:
from pathlib import Path
import subprocess

SAMPLE_REPO = Path("/content/people-counting-samples")

if not SAMPLE_REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/yelaman1x9/people-counting.git",
        str(SAMPLE_REPO)
    ], check=True)

script = SAMPLE_REPO / "scripts/download_samples.py"

assert script.is_file(), "Sample download script missing"

print("Sample source ready:", SAMPLE_REPO)
print("\nDownload script:")
print(script.read_text(encoding="utf-8")[:12000])

In [ ]:

from pathlib import Path
import requests
import cv2
from IPython.display import display, Image

VIDEO_DIR = OUTPUT_DIR / "videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

STREET_VIDEO = VIDEO_DIR / "street_walk.mp4"

VIDEO_URL = (
    "https://videos.pexels.com/video-files/"
    "3552510/3552510-hd_1920_1080_30fps.mp4"
)

# Download directly to Google Drive
if not STREET_VIDEO.is_file():
    temp = STREET_VIDEO.with_suffix(".part")

    with requests.get(
        VIDEO_URL, stream=True, timeout=120
    ) as response:
        response.raise_for_status()

        with open(temp, "wb") as f:
            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    f.write(chunk)

    temp.replace(STREET_VIDEO)

print("Downloaded:", STREET_VIDEO)
print(
    "Size MB:",
    round(STREET_VIDEO.stat().st_size / 1024**2, 2)
)

# Inspect video
cap = cv2.VideoCapture(str(STREET_VIDEO))

assert cap.isOpened(), "Cannot open video"

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

ok, frame = cap.read()
cap.release()

assert ok, "Cannot read first frame"
assert fps > 0 and width > 0 and height > 0

# Proposed vertical counting line
REGION_STREET = [
    (width // 2, 0),
    (width // 2, height - 1)
]

preview = frame.copy()

cv2.line(
    preview,
    REGION_STREET[0],
    REGION_STREET[1],
    (0, 0, 255),
    5
)

# Resize preview for Colab
preview = cv2.resize(
    preview,
    (960, int(height * 960 / width))
)

print("\n===== STREET WALK VIDEO =====")
print("Resolution:", width, "x", height)
print("FPS:", fps)
print("Frames:", frames)
print("Counting Line:", REGION_STREET)

display(
    Image(data=cv2.imencode(".jpg", preview)[1].tobytes())
)

print("\nSTREET VIDEO PREPARATION: PASS")


In [ ]:

from ultralytics import solutions
from pathlib import Path
import cv2
import json
import torch

RUN_STREET = OUTPUT_DIR / "street_walk_counting"
RUN_STREET.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = RUN_STREET / "street_walk_counted.mp4"
REPORT = RUN_STREET / "street_walk_results.json"

assert STREET_VIDEO.is_file()
assert not OUTPUT_VIDEO.exists()
assert not REPORT.exists()

cap = cv2.VideoCapture(str(STREET_VIDEO))
assert cap.isOpened()

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

region = [
    (width // 2, 0),
    (width // 2, height - 1)
]

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

assert writer.isOpened()

counter = solutions.ObjectCounter(
    model="yolo26n.pt",
    region=region,
    tracker="bytetrack.yaml",
    classes=[0],
    conf=0.25,
    imgsz=640,
    device=0 if torch.cuda.is_available() else "cpu",
    show=False,
    verbose=False
)

frames_processed = 0
last_result = None

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        last_result = counter(frame)
        writer.write(last_result.plot_im)
        frames_processed += 1

finally:
    cap.release()
    writer.release()

assert frames_processed > 0
assert last_result is not None
assert OUTPUT_VIDEO.is_file()
assert OUTPUT_VIDEO.stat().st_size > 0

summary = {
    "video": str(STREET_VIDEO),
    "model": "yolo26n.pt",
    "tracker": "bytetrack.yaml",
    "target_class": "person",
    "fps": fps,
    "processed_frames": frames_processed,
    "counting_line": region,
    "in_count": int(last_result.in_count),
    "out_count": int(last_result.out_count),
    "total_crossings": (
        int(last_result.in_count)
        + int(last_result.out_count)
    ),
    "output_video": str(OUTPUT_VIDEO)
}

REPORT.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("\n========== STREET WALK COUNTING ==========")
print("Processed frames:", frames_processed)
print("IN:", summary["in_count"])
print("OUT:", summary["out_count"])
print("Total crossings:", summary["total_crossings"])
print("Output video:", OUTPUT_VIDEO)
print("Report:", REPORT)
print("\nSTREET WALK COUNTING: PASS")


In [ ]:

from pathlib import Path
from IPython.display import Video, display
import subprocess

RUN_STREET = OUTPUT_DIR / "street_walk_counting"

SOURCE = RUN_STREET / "street_walk_counted.mp4"
PREVIEW = RUN_STREET / "street_walk_preview_h264.mp4"

assert SOURCE.is_file(), "Counting output not found"

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(SOURCE),
        "-vf", "scale=960:-2",
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "23",
        "-pix_fmt", "yuv420p",
        "-an",
        str(PREVIEW)
    ],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

assert PREVIEW.is_file()
assert PREVIEW.stat().st_size > 0

print("Preview saved:", PREVIEW)
print(
    "Size MB:",
    round(PREVIEW.stat().st_size / 1024**2, 2)
)

display(Video(str(PREVIEW), embed=True))


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

BASE = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/object_counting"
)

DEMO = BASE / "person_counting_demo"
STREET = BASE / "street_walk_counting"

required = {
    "demo_report": DEMO / "counting_results.json",
    "demo_video": DEMO / "counted_people_h264.mp4",
    "street_report": STREET / "street_walk_results.json",
    "street_video": STREET / "street_walk_preview_h264.mp4",
}

for name, path in required.items():
    assert path.is_file(), f"Missing: {name}"
    print("OK:", name)

demo = json.loads(required["demo_report"].read_text())
street = json.loads(required["street_report"].read_text())

assert demo["processed_frames"] == 62
assert street["processed_frames"] == 712

checkpoint = {
    "task": "02 - Object Counting",
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "technical_tests_passed_manual_review_pending",
    "model": "yolo26n.pt",
    "tracker": "bytetrack.yaml",
    "class": "person",
    "experiments": {
        "short_demo": {
            "frames": demo["processed_frames"],
            "in": demo["in_count"],
            "out": demo["out_count"]
        },
        "street_walk": {
            "frames": street["processed_frames"],
            "in": street["in_count"],
            "out": street["out_count"]
        }
    },
    "pending": [
        "Manual verification of crossing counts",
        "Finalize notebook documentation",
        "Review and commit code to GitHub"
    ],
    "next_task": "03 - VisionEye Mapping"
}

path = BASE / "TASK2_PROGRESS_CHECKPOINT.json"
path.write_text(
    json.dumps(checkpoint, indent=2),
    encoding="utf-8"
)

print("\nTASK 2 CHECKPOINT: SAVED")
print(path)

## Reproducibility setup
Run after dependency installation. Fixed seeds reduce variability but do not guarantee identical results across devices. Save dependency versions, hardware, source video identity, dataset versions, and final configuration with each future experiment.

## 1. Video and counting-line configuration

Two experiments were performed:

- Short demo: 640x360, 62 frames, horizontal counting line
  from `(0, 180)` to `(639, 180)`.
- Street Walk: 1920x1080, 712 frames, vertical counting line
  from `(960, 0)` to `(960, 1079)`.

Both experiments targeted the COCO `person` class (ID 0).


## 2. Pretrained model and ObjectCounter

The experiments used:

- Pretrained checkpoint: `yolo26n.pt`
- Ultralytics: 8.4.208
- Tracker: `bytetrack.yaml`
- Confidence threshold: 0.25
- Solution: `solutions.ObjectCounter`
- Display mode: `show=False`, compatible with Google Colab

The model was used for inference; no additional training was done.


## 3. Tracking and video processing

Each video was processed frame by frame using OpenCV.
A persistent ObjectCounter instance maintained tracking and
line-crossing counts.

Annotated frames were written to output videos, and H.264
preview versions were generated for browser playback.
Results were also saved in JSON reports on Google Drive.


## 4. Counting validation and limitations

The system reports line-crossing events, not the number of
unique people appearing anywhere in the video.

Tracking errors, occlusion, crowded scenes and ID switches
may affect the results.

Manual ground-truth verification of the crossing counts
is still pending. Therefore, counting accuracy is not
claimed or estimated from these experiments.


## 5. Results and conclusions

| Experiment | Frames | IN | OUT | Total crossings |
|---|---:|---:|---:|---:|
| Short Demo | 62 | 2 | 2 | 4 |
| Street Walk | 712 | 2 | 6 | 8 |

Both technical experiments completed successfully.
The output videos show tracked bounding boxes, IDs,
counting lines and changing IN/OUT counters.

The IN/OUT labels describe crossing directions for the
configured line, not necessarily entry into or exit from
a physical building.

### References

- [Ultralytics Object Counting](https://docs.ultralytics.com/guides/object-counting/)
- [Ultralytics Models](https://docs.ultralytics.com/models/)
